In [4]:
import os
import json
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from firecrawl import FirecrawlApp

# 1. Load Environment Variables
load_dotenv(override=True)

# 2. Initialize Firecrawl
app = FirecrawlApp(api_key=os.getenv("FIRECRAWL_API_KEY"))

# 3. Initialize the Groq LLM Client (THIS IS WHAT WAS MISSING!)
llm_client = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2, 
    api_key=os.getenv("GROQ_API_KEY")
)

# 4. The URLs you want to process
target_urls = [
    'http://www.valmikiramayan.net/bala_1.htm',
    'http://www.valmikiramayan.net/bala_2.htm'
]

master_graph_data = []

# 5. The Extraction Prompt
extraction_prompt = """
You are a strict backend data extraction API. 
Read the following text containing translated Sanskrit verses.
Extract the main event into the exact JSON schema provided.
OUTPUT ONLY VALID JSON. NO PREAMBLE. NO CONVERSATIONAL TEXT. NO MARKDOWN.

SCHEMA:
{
  "source_url": "string",
  "entities": [
    {"name": "string", "type": "string", "karaka_role": "string"}
  ],
  "relation": "string"
}

TEXT TO ANALYZE:
"""

# 6. The Automation Loop
for url in target_urls:
    print(f"Scraping {url}...")
    
    # A. Scrape the raw text (Updated to dict access for safety)
    scraped_data = app.scrape_url(url, formats= ['markdown'])
    raw_markdown = scraped_data.markdown or ""
    
    # B. Truncate text to stay under Groq's free tier limits
    safe_markdown = raw_markdown[:5000] 
    
    # C. Send the SAFE truncated text to the LLM
    print("Sending a safe snippet to Groq...")
    response = llm_client.invoke(extraction_prompt + safe_markdown)
    
    # D. Convert the LLM text output into a Python dictionary
    # We use .strip() to clean up any hidden characters the LLM might add
    try:
        raw_output = response.content
        
        # PYTHON HACK: Find the first '{' and the last '}' to ignore any chatty text
        start_index = raw_output.find('{')
        end_index = raw_output.rfind('}') + 1
        
        if start_index != -1 and end_index != 0:
            clean_text = raw_output[start_index:end_index]
            structured_json = json.loads(clean_text)
            master_graph_data.append(structured_json)
            print("Successfully extracted JSON!")
        else:
            raise ValueError("No JSON structure found.")
            
    except Exception as e:
        # Let's print exactly what the LLM said so we can debug if it fails again!
        print(f"Error parsing JSON from {url}. Here is what the LLM actually said:")
        print("--- RAW LLM OUTPUT ---")
        print(response.content)
        print("----------------------")

# 7. Save the final output to a file
with open('automated_graph_data.json', 'w') as f:
    json.dump(master_graph_data, f, indent=2)

print("\nExtraction complete! Check 'automated_graph_data.json'. Ready to send to Neo4j.")

Scraping http://www.valmikiramayan.net/bala_1.htm...
Sending a safe snippet to Groq...
Successfully extracted JSON!
Scraping http://www.valmikiramayan.net/bala_2.htm...
Sending a safe snippet to Groq...
Successfully extracted JSON!

Extraction complete! Check 'automated_graph_data.json'. Ready to send to Neo4j.
